# Transfer Learning (MobileNet)

## Data Preparation

In [1]:
from tensorflow.keras.datasets import cifar10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
class_names = [
    'airplane', 
    'automobile', 
    'bird', 
    'cat', 
    'deer', 
    'dog', 
    'frog', 
    'horse',
    'ship', 
    'truck',
]

In [2]:
from tensorflow.keras.applications.mobilenet import preprocess_input
x_train = preprocess_input(x_train)
x_test = preprocess_input(x_test)

## Data Augmentation

In [3]:
from tensorflow.keras import layers, Sequential
data_augmentation = Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
    layers.RandomCrop(32, 32),
])

## Label Encoding

In [4]:
from tensorflow.keras.utils import to_categorical
y_train_encoded = to_categorical(y_train, num_classes=len(class_names))
y_test_encoded = to_categorical(y_test, num_classes=len(class_names))

## Building and Training the Model

In [5]:
from tensorflow.keras.applications.mobilenet import MobileNet
from tensorflow.keras import layers, Sequential

original_shape = x_train.shape[1:]
upscaled_shape = (128, 128, 3) # x3

pretrained_model = MobileNet(weights='imagenet', include_top=False, input_shape=upscaled_shape)
for layer in pretrained_model.layers:
    layer.trainable = False

feature_learning = Sequential([
    layers.Input(shape=original_shape),
    layers.Resizing(upscaled_shape[0], upscaled_shape[1]),
    pretrained_model,
])

classification = Sequential([
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(1024),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.75),
    layers.Dense(len(class_names), activation='softmax'),
])

model = Sequential([
    feature_learning,
    classification,
])

In [6]:
from tensorflow import device
with device('/GPU'):
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(x_train, y_train_encoded, epochs=5, batch_size=64, validation_split=0.2)

Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 53s 82ms/step - accuracy: 0.7235 - loss: 0.9105 - val_accuracy: 0.8764 - val_loss: 0.3626
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 52s 83ms/step - accuracy: 0.8477 - loss: 0.4373 - val_accuracy: 0.8823 - val_loss: 0.3488
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 52s 83ms/step - accuracy: 0.8647 - loss: 0.3860 - val_accuracy: 0.8869 - val_loss: 0.3353
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 50s 81ms/step - accuracy: 0.8767 - loss: 0.3576 - val_accuracy: 0.8904 - val_loss: 0.3295
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 50s 80ms/step - accuracy: 0.8885 - loss: 0.3229 - val_accuracy: 0.8948 - val_loss: 0.3229


In [7]:
feature_learning.layers.insert(1, data_augmentation)
for layer in pretrained_model.layers[-6:]:
    layer.trainable = True
display(pretrained_model.summary(show_trainable=True))

Model: "mobilenet_1.00_128"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)                ┃ Output Shape          ┃    Param # ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ input_layer (InputLayer)    │ (None, 128, 128, 3)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ conv1 (Conv2D)              │ (None, 64, 64, 32)    │        864 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ conv1_bn                    │ (None, 64, 64, 32)    │        128 │   N   │
│ (BatchNormalization)        │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ conv1_relu (ReLU)           │ (None, 64, 64, 32)    │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ conv_dw_1 (DepthwiseConv2D) │ (None, 64, 64, 32)    │        288 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ conv_dw_1_bn                │ (None, 64, 64, 32)    │        128 │   N   │
│ (BatchNormalization)        │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ conv_dw_1_relu (ReLU)       │ (None, 64, 64, 32)    │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ conv_pw_1 (Conv2D)          │ (None, 64, 64, 64)    │      2,048 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ conv_pw_1_bn                │ (None, 64, 64, 64)    │        256 │   N   │
│ (BatchNormalization)        │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ conv_pw_1_relu (ReLU)       │ (None, 64, 64, 64)    │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ conv_pad_2 (ZeroPadding2D)  │ (None, 65, 65, 64)    │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ conv_dw_2 (DepthwiseConv2D) │ (None, 32, 32, 64)    │        576 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ conv_dw_2_bn                │ (None, 32, 32, 64)    │        256 │   N   │
│ (BatchNormalization)        │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ conv_dw_2_relu (ReLU)       │ (None, 32, 32, 64)    │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ conv_pw_2 (Conv2D)          │ (None, 32, 32, 128)   │      8,192 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ conv_pw_2_bn                │ (None, 32, 32, 128)   │        512 │   N   │
│ (BatchNormalization)        │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ conv_pw_2_relu (ReLU)       │ (None, 32, 32, 128)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ conv_dw_3 (DepthwiseConv2D) │ (None, 32, 32, 128)   │      1,152 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ conv_dw_3_bn                │ (None, 32, 32, 128)   │        512 │   N   │
│ (BatchNormalization)        │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ conv_dw_3_relu (ReLU)       │ (None, 32, 32, 128)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ conv_pw_3 (Conv2D)          │ (None, 32, 32, 128)   │     16,384 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ conv_pw_3_bn                │ (None, 32, 32, 128)   │        512 │   N 

 Total params: 3,228,864 (12.32 MB)

 Trainable params: 1,061,888 (4.05 MB)

 Non-trainable params: 2,166,976 (8.27 MB)

None

In [8]:
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.000001)
optimizer = AdamW(learning_rate=0.00005, weight_decay=0.00001)

with device('/GPU'):
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(x_train, y_train_encoded, epochs=20, batch_size=64, callbacks=[reduce_lr], validation_split=0.2)

Epoch 1/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 61s 95ms/step - accuracy: 0.8797 - loss: 0.3384 - val_accuracy: 0.8983 - val_loss: 0.3103 - learning_rate: 5.0000e-05
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 58s 93ms/step - accuracy: 0.9058 - loss: 0.2663 - val_accuracy: 0.9037 - val_loss: 0.3005 - learning_rate: 5.0000e-05
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 58s 93ms/step - accuracy: 0.9205 - loss: 0.2258 - val_accuracy: 0.9034 - val_loss: 0.2954 - learning_rate: 5.0000e-05
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 59s 94ms/step - accuracy: 0.9320 - loss: 0.1951 - val_accuracy: 0.9058 - val_loss: 0.2907 - learning_rate: 5.0000e-05
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 58s 93ms/step - accuracy: 0.9379 - loss: 0.1731 - val_accuracy: 0.9079 - val_loss: 0.2819 - learning_rate: 5.0000e-05
Epoch 6/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 60s 96ms/step - accuracy: 0.9477 - loss: 0.1514 - val_accuracy: 0.9075 - val_loss: 0.2820 - learning_rate: 5.0000e-05
Epoch 7/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 59s 94ms/ste

## Result

In [9]:
from sklearn.metrics import classification_report
import numpy as np
with device('/GPU'):
    y_pred_probs = model.predict(x_test, verbose=False)
    y_pred_labels = np.argmax(y_pred_probs, axis=1)
    y_true_labels = np.argmax(y_test_encoded, axis=1)
    print(classification_report(y_true_labels, y_pred_labels, target_names=class_names))

              precision    recall  f1-score   support

    airplane       0.92      0.91      0.91      1000
  automobile       0.95      0.95      0.95      1000
        bird       0.89      0.89      0.89      1000
         cat       0.84      0.83      0.83      1000
        deer       0.88      0.90      0.89      1000
         dog       0.85      0.86      0.86      1000
        frog       0.93      0.93      0.93      1000
       horse       0.93      0.92      0.92      1000
        ship       0.93      0.95      0.94      1000
       truck       0.94      0.94      0.94      1000

    accuracy                           0.91     10000
   macro avg       0.91      0.91      0.91     10000
weighted avg       0.91      0.91      0.91     10000

